<a href="https://colab.research.google.com/github/daniamanal/Global_Space_Mission_Analysis/blob/main/Global_Space_Missions_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

<center><img src="https://i.imgur.com/9hLRsjZ.jpg" height=400></center>

This dataset was scraped from [nextspaceflight.com](https://nextspaceflight.com/launches/past/?page=1) and includes all the space missions since the beginning of Space Race between the USA and the Soviet Union in 1957!

### Install Package with Country Codes

In [110]:
%pip install iso3166

### Upgrade Plotly

Run the cell below if you are working with Google Colab.

In [111]:
%pip install --upgrade plotly

### Import Statements

In [112]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
from iso3166 import countries
from datetime import datetime, timedelta

### Notebook Presentation

In [113]:
pd.options.display.float_format = '{:,.2f}'.format

### Load the Data

In [114]:
df_data = pd.read_csv('mission_launches.csv')

# Preliminary Data Exploration

* What is the shape of `df_data`?
* How many rows and columns does it have?
* What are the column names?
* Are there any NaN values or duplicates?

In [115]:
df_data.shape

(4324, 9)

In [116]:
df_data.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'Organisation', 'Location', 'Date',
       'Detail', 'Rocket_Status', 'Price', 'Mission_Status'],
      dtype='object')

In [117]:
df_data.isna().sum()

,0
Unnamed: 0.1,0
Unnamed: 0,0
Organisation,0
Location,0
Date,0
Detail,0
Rocket_Status,0
Price,3360
Mission_Status,0


## Data Cleaning - Check for Missing Values and Duplicates

Consider removing columns containing junk data.

In [118]:
df_data.duplicated().sum()

np.int64(0)

In [119]:
df_data.isnull().sum()

,0
Unnamed: 0.1,0
Unnamed: 0,0
Organisation,0
Location,0
Date,0
Detail,0
Rocket_Status,0
Price,3360
Mission_Status,0


## Descriptive Statistics

In [120]:
df_data.describe()

,Unnamed: 0.1,Unnamed: 0
count,"4,324.00","4,324.00"
mean,"2,161.50","2,161.50"
std,"1,248.38","1,248.38"
min,0.00,0.00
25%,"1,080.75","1,080.75"
50%,"2,161.50","2,161.50"
75%,"3,242.25","3,242.25"
max,"4,323.00","4,323.00"


# Number of Launches per Company

Create a chart that shows the number of space mission launches by organisation.

In [121]:
df_launches = df_data['Organisation'].value_counts().reset_index()
df_launches.columns = ['Organisation','Launch_Count']

In [122]:
fig=px.bar(df_launches,
           x='Launch_Count',
           y='Organisation',
           labels={'Launch_Count':'Launch Count'},
           title='Number of space mission launches by organisation')
fig.show()

# Number of Active versus Retired Rockets

How many rockets are active compared to those that are decomissioned?

In [123]:
df_data['Rocket_Status'].value_counts()

,count
Rocket_Status,
StatusRetired,3534
StatusActive,790


# Distribution of Mission Status

How many missions were successful?
How many missions failed?

In [124]:
df_data['Mission_Status'].value_counts()

,count
Mission_Status,
Success,3879
Failure,339
Partial Failure,102
Prelaunch Failure,4


# How Expensive are the Launches?

Create a histogram and visualise the distribution. The price column is given in USD millions (careful of missing values).

In [125]:
df_price = df_data[['Price']].copy().dropna() #copy the Price column and turn it to a df
df_price['Price']=pd.to_numeric(df_price['Price'], errors='coerce')
# df_price['Price'] = pd.to_numeric(df_data['Price'], errors='coerce')
# df_price.reset_index()

In [126]:
fig = px.histogram(df_price,
                   x="Price",
                   nbins=30,
                   labels={"Price": "Price(USD millions),"},
                   title="Distribution of Launch Prices")
fig.show()

# Use a Choropleth Map to Show the Number of Launches by Country

* Create a choropleth map using [the plotly documentation](https://plotly.com/python/choropleth-maps/)
* Experiment with [plotly's available colours](https://plotly.com/python/builtin-colorscales/). I quite like the sequential colour `matter` on this map.
* You'll need to extract a `country` feature as well as change the country names that no longer exist.

Wrangle the Country Names

You'll need to use a 3 letter country code for each country. You might have to change some country names.

* Russia is the Russian Federation
* New Mexico should be USA
* Yellow Sea refers to China
* Shahrud Missile Test Site should be Iran
* Pacific Missile Range Facility should be USA
* Barents Sea should be Russian Federation
* Gran Canaria should be USA


You can use the iso3166 package to convert the country names to Alpha3 format.

In [127]:
#create a country column in a new df
df_map = df_data[['Location']].copy().dropna() #copy Location column, clean it and convert to a df
df_map['Country']= df_map['Location'].str.split(',') # split the values after comma
df_map['Country']= df_map['Country'].str[-1] #pick the last value
df_map['Country']= df_map['Country'].str.strip() #strip spaces

In [128]:
#rename old country names
df_map['Country'] = df_map['Country'].replace({
    'Russia': 'Russian Federation',
    'New Mexico': 'USA',
    'Yellow Sea': 'China',
    'Shahrud Missile Test Site': 'Iran',
    'Pacific Missile Range Facility': 'USA',
    'Barents Sea': 'Russian Federation',
    'Gran Canaria': 'USA'
})

In [129]:
# create iso_code column
def get_iso3(country_name): # function to convert country names to iso3 country codes
  try:
    return countries.get(country_name).alpha3
  except:
    return None
df_map['ISO_Code'] = df_map['Country'].apply(get_iso3) # Convert Country column avlues to ISO-3 Country Codes and store them in new column

In [130]:
# Drop rows where conversion failed
df_map = df_map.dropna(subset=['ISO_Code'])

In [131]:
# Create launch count df to store launches per country
launch_counts = df_map.groupby(['Country', 'ISO_Code']).size() #count launches per country
launch_counts=launch_counts.reset_index()# turn index to column
launch_counts.columns = ['Country', 'ISO_Code', 'Launch_Count'] # rename columns

In [132]:
# Create Choropleth Map
fig = px.choropleth(
    launch_counts,
    locations='ISO_Code',
    color='Launch_Count',
    hover_name='Country',
    color_continuous_scale='matter',
    title='Number of Space Launches by Country'
)
fig.show()

# Use a Choropleth Map to Show the Number of Failures by Country


In [133]:
# Filter the failed missions from df_data
df_failure = df_data[df_data['Mission_Status']=='Failure'].copy()

In [134]:
# create and clean country column
df_failure['Country']=df_failure['Location'].str.split(',').str[-1].str.strip()
# Fix Special Cases
df_failure['Country'] = df_failure['Country'].replace({
    'Russia': 'Russian Federation',
    'New Mexico': 'USA',
    'Yellow Sea': 'China',
    'Shahrud Missile Test Site': 'Iran',
    'Pacific Missile Range Facility': 'USA',
    'Barents Sea': 'Russian Federation',
    'Gran Canaria': 'USA'
})

In [135]:
# Convert to ISO-3 Codes
def get_iso3(country_name):
  try:
    return countries.get(country_name).alpha3
  except:
      return None
df_failure['ISO_Code'] = df_failure['Country'].apply(get_iso3)

# Drop failed conversions
df_failure = df_failure.dropna(subset=['ISO_Code'])

In [136]:
# create a df to store failure counts per country
fail_counts = df_failure.groupby(['Country', 'ISO_Code']).size().reset_index(name='Failure_Count')

In [137]:
#plot choropleth to visualize fail counts per country
fig = px.choropleth(
    fail_counts,
    locations='ISO_Code',
    color='Failure_Count',
    hover_name='Country',
    color_continuous_scale='inferno',
    title='Number of Failed Space Missions by Country'
)

fig.show()

# Create a Plotly Sunburst Chart of the countries, organisations, and mission status.

In [138]:
# Prepare data for same ountry wrangling
df_sun = df_data.copy()
df_sun['Country'] = df_sun['Location'].str.split(',').str[-1].str.strip()
df_sun['Country'] = df_sun['Country'].replace({
    'Russia': 'Russian Federation',
    'New Mexico': 'USA',
    'Yellow Sea': 'China',
    'Shahrud Missile Test Site': 'Iran',
    'Pacific Missile Range Facility': 'USA',
    'Barents Sea': 'Russian Federation',
    'Gran Canaria': 'USA'
})

In [139]:
# Clean relevant columns
df_sun = df_sun[['Country', 'Organisation', 'Mission_Status']].dropna()

In [140]:
# Create Sunburst chart
fig= px.sunburst(
    df_sun, path=['Country', 'Organisation', 'Mission_Status'],
    title='Space Missions Hierarchy: Country → Organisation → Mission Status'
)
fig.show()
# Center → Countries, Middle ring → Organisations, Outer ring → Mission status

# Analyse the Total Money Spent by Organisation on Space Missions

In [141]:
df_money = df_data[['Organisation', 'Price']].copy()
# Convert df_money to numeric
df_money['Price'] = pd.to_numeric(df_money['Price'], errors='coerce')
# remove missing values
df_money= df_money.dropna()

In [142]:
# Collect total spending per organisation
org_spending = df_money.groupby('Organisation')['Price'].sum().reset_index()
org_spending = org_spending.rename(columns={'Price': 'Total_Spending'})
org_spending = org_spending.sort_values(by='Total_Spending', ascending=False)

In [143]:
fig = px.bar(
    org_spending,
    x='Organisation',
    y='Total_Spending',
    title='Total Money Spent by Organisation on Space Missions',
    labels={'Total_Spending': 'Total Spending (USD Millions)'}
)
# fig.update_layout(xaxis_tickangle=-45)
fig.show()

# Analyse the Amount of Money Spent by Organisation per Launch

Compute Mean Cost Per Launch


In [144]:
cost_per_launch = df_money.groupby('Organisation')['Price'].mean().reset_index()
cost_per_launch = cost_per_launch.rename(columns={'Price':'Avg_Cost_per_Lanuch'})

Add Launch Counts

In [145]:
launch_counts = df_money['Organisation'].value_counts().reset_index()
launch_counts.columns = ['Organisation', 'Launch_Count']
# merge
cost_per_launch = cost_per_launch.merge(launch_counts, on='Organisation')
# Sort
cost_per_launch = cost_per_launch.sort_values(by='Avg_Cost_per_Lanuch')

In [146]:
fig = px.bar(
    cost_per_launch,
    x='Organisation',
    y='Avg_Cost_per_Lanuch',
    title='Average Cost per Launch by Organisation',
    labels={'Avg_Cost_per_Lanuch': 'Avg Cost (USD Millions)'},
    hover_data=['Launch_Count']
)
fig.show()

# Chart the Number of Launches per Year

In [147]:
# Convert to data to time
df_time = df_data.copy()
df_time['Date'] = pd.to_datetime(df_time['Date'], errors='coerce')
# drop invalid dates from Date column
df_time = df_time.dropna(subset='Date')

In [148]:
# Extract the Year from Date
df_time['Year']=df_time['Date'].dt.year

In [149]:
# Count launches per year
launches_per_year = df_time['Year'].value_counts().reset_index().sort_index()
#rename columns
launches_per_year.columns = ['Year', 'Launch_Count']

In [150]:
fig = px.bar(launches_per_year,
             x='Year',
             y='Launch_Count',
             title= 'Number of Space Launches per Year',)
fig.show()

# Chart the Number of Launches Month-on-Month until the Present

Which month has seen the highest number of launches in all time? Superimpose a rolling average on the month on month time series chart.

In [151]:
# Extract the Year Month from Date
df_time['YearMonth'] = df_time['Date'].dt.to_period('M')
# Count launches per month
monthly_launches = df_time.groupby('YearMonth').size().reset_index(name='Launch_Count')
#convert back to datetime for plotly
monthly_launches['YearMonth'] = monthly_launches['YearMonth'].astype(str)
monthly_launches['YearMonth'] = pd.to_datetime(monthly_launches['YearMonth'])

/tmp/ipykernel_703/251320976.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_time['YearMonth'] = df_time['Date'].dt.to_period('M')


In [152]:
fig = px.bar(monthly_launches,
             x='YearMonth',
             y='Launch_Count',
             title= 'Number of Space Launches Month-on-Month',)
fig.show()

In [153]:
# Rolling Average
monthly_launches['Rolling_Avg'] = monthly_launches['Launch_Count'].rolling(6).mean()

fig = px.line(
    monthly_launches,
    x='YearMonth',
    y=['Launch_Count', 'Rolling_Avg'],
    title='Monthly Launches with Trend'
)

fig.show()

# Launches per Month: Which months are most popular and least popular for launches?

Some months have better weather than others. Which time of year seems to be best for space missions?

In [154]:
# Extract month and month names
df_time['Month'] = df_time['Date'].dt.month
df_time['Month_Name'] = df_time['Date'].dt.month_name()

In [155]:
# count & sort monthly launches
monthly_counts = df_time.groupby(['Month', 'Month_Name']).size().reset_index(name='Launch_Count').sort_values('Month')


In [156]:
fig = px.bar(monthly_counts,
             x='Month_Name',
             y='Launch_Count',
             title='Number of Space Launches by Month',
             labels={'Launch_Count':'Number of Launches'})
fig.show()

In [157]:
most_popular_month = monthly_counts.loc[monthly_counts['Launch_Count'].idxmax()]
least_popular_month = monthly_counts.loc[monthly_counts['Launch_Count'].idxmin()]

In [158]:
most_popular_month
least_popular_month

,0
Month,1
Month_Name,January
Launch_Count,265


# How has the Launch Price varied Over Time?

Create a line chart that shows the average price of rocket launches over time.

In [159]:
# clean the price column
df_time['Price'] = pd.to_numeric(df_time['Price'], errors='coerce')
df_time['Year'] = pd.to_numeric(df_time['Year'], errors='coerce')
df_time=df_time.dropna(subset=['Date','Price'])


In [160]:
avg_price_per_year = df_time.groupby('Year')['Price'].mean().reset_index()
avg_price_per_year.columns = ['Year','Avg_Price']

In [161]:
fig = px.line(
    avg_price_per_year,
    x='Year',
    y='Avg_Price',
    title = 'Average Rocket Launch Price Over Time',
    labels={'Avg_Price':'Average Price (USD Millions)'}
)
fig.show()

In [162]:
avg_price_per_year['Rolling_avg']=avg_price_per_year['Avg_Price'].rolling(5).mean()
fig=px.line(avg_price_per_year,
            y='Rolling_avg',
            x='Year',
            title='Average Launch Price with Trend')
fig.show()

The average cost of rocket launches has decreased over time, likely because of advancements in technology. However, occasional spikes suggest the presence of high-cost missions.

# Chart the Number of Launches over Time by the Top 10 Organisations.

How has the dominance of launches changed over time between the different players?

In [163]:
# Find top ten organisations
top_orgs = df_time['Organisation'].value_counts().sort_values(ascending=False).head(10).index
df_top = df_time[df_time['Organisation'].isin(top_orgs)]

In [164]:

# Count launches per organization
# launch_counts = df_data['Organisation'].value_counts().reset_index()
# launch_counts.columns = ['Organisation', 'Launch_Count']
launch_trends = df_top.groupby(['Year', 'Organisation']).size().reset_index(name='Launch_Count')
launch_trends

,Year,Organisation,Launch_Count
0,1964,US Air Force,2
1,1965,US Air Force,2
2,1966,US Air Force,3
3,1967,US Air Force,6
4,1968,US Air Force,8
...,...,...,...
188,2020,MHI,2
189,2020,Northrop,2
190,2020,SpaceX,13
191,2020,ULA,4


In [165]:
import plotly.express as px

fig = px.line(
    launch_trends,
    x='Year',
    y='Launch_Count',
    color='Organisation',
    title='Number of Launches Over Time by Top 10 Organisations',
    markers=True
)

fig.show()

# Cold War Space Race: USA vs USSR

The cold war lasted from the start of the dataset up until 1991.

## Create a Plotly Pie Chart comparing the total number of launches of the USSR and the USA

Hint: Remember to include former Soviet Republics like Kazakhstan when analysing the total number of launches.

In [166]:
# Extract Country
df_pie = df_data.copy()
df_pie['Country'] = df_pie['Location'].str.split(',').str[-1].str.strip()

In [167]:
# Map Countries to USSR vs USA
def map_country(country):
    if country in ['USA']:
        return 'USA'
    elif country in ['Russia', 'Russian Federation', 'Kazakhstan']:
        return 'USSR'
    else:
        return 'Other'
df_pie['Superpower'] = df_pie['Country'].apply(map_country)

In [168]:
# Filter Only USSR vs USA
df_pie = df_pie[df_pie['Superpower'].isin(['USA', 'USSR'])]

In [169]:
# Count Launches
launch_counts = df_pie['Superpower'].value_counts().reset_index()
launch_counts.columns = ['Superpower', 'Launch_Count']

In [170]:
# Plot Pie Chart
fig = px.pie(
    launch_counts,
    names='Superpower',
    values='Launch_Count',
    title='Total Number of Launches: USA vs USSR',
    color='Superpower',
    color_discrete_map={
        'USA': 'blue',
        'USSR': 'red'
    }
)

fig.show()

## Create a Chart that Shows the Total Number of Launches Year-On-Year by the Two Superpowers

In [171]:
# Prepare Data
df_super = df_data.copy()

# Clean Date
df_super['Date'] = pd.to_datetime(df_super['Date'], errors='coerce')
df_super = df_super.dropna(subset=['Date'])

# Extract Year
df_super['Year'] = df_super['Date'].dt.year

# Extract Country
df_super['Country'] = df_super['Location'].str.split(',').str[-1].str.strip()

In [172]:
# Map to Superpowers
def map_country(country):
    if country == 'USA':
        return 'USA'
    elif country in ['Russia', 'Russian Federation', 'Kazakhstan']:
        return 'USSR'
    else:
        return 'Other'

df_super['Superpower'] = df_super['Country'].apply(map_country)

In [173]:
# Filter Only USA & USSR
df_super = df_super[df_super['Superpower'].isin(['USA', 'USSR'])]

In [174]:
# Count Launches per Year
launches_yearly = df_super.groupby(['Year', 'Superpower']).size().reset_index(name='Launch_Count')

In [175]:
#Plot
fig = px.line(
    launches_yearly,
    x='Year',
    y='Launch_Count',
    color='Superpower',
    title='Yearly Space Launches: USA vs USSR',
    markers=True
)

fig.show()

## Chart the Total Number of Mission Failures Year on Year.

In [176]:
# Prepare Data
df_fail_time = df_data.copy()

# Convert Date
df_fail_time['Date'] = pd.to_datetime(df_fail_time['Date'], errors='coerce')
df_fail_time = df_fail_time.dropna(subset=['Date'])

# Extract Year
df_fail_time['Year'] = df_fail_time['Date'].dt.year

In [177]:
# Filter Failed Missions
df_fail_time = df_fail_time[df_fail_time['Mission_Status'] != 'Success']

In [178]:
# Count Failures per Year
failures_per_year = df_fail_time.groupby('Year').size().reset_index(name='Failure_Count')

In [179]:
# Plot with Plotly
fig = px.line(
    failures_per_year,
    x='Year',
    y='Failure_Count',
    title='Number of Mission Failures Year-on-Year',
    markers=True
)

fig.show()

## Chart the Percentage of Failures over Time

Did failures go up or down over time? Did the countries get better at minimising risk and improving their chances of success over time?

In [180]:
#Prepare Data
df_rate = df_data.copy()

# Clean Date
df_rate['Date'] = pd.to_datetime(df_rate['Date'], errors='coerce')
df_rate = df_rate.dropna(subset=['Date'])

# Extract Year
df_rate['Year'] = df_rate['Date'].dt.year

In [181]:
# create failure indicator
df_rate['Failure'] = df_rate['Mission_Status'] != 'Success'

In [182]:
# Compute Total Launches & Failures per Year
yearly_stats = df_rate.groupby('Year').agg(
    Total_Launches=('Mission_Status', 'count'),
    Failures=('Failure', 'sum')
).reset_index()

In [183]:
# Calculate Failure Percentage
yearly_stats['Failure_Percentage'] = (
    yearly_stats['Failures'] / yearly_stats['Total_Launches']
) * 100

In [184]:
# Plot Line Chart

fig = px.line(
    yearly_stats,
    x='Year',
    y='Failure_Percentage',
    title='Percentage of Mission Failures Over Time',
    labels={'Failure_Percentage': 'Failure (%)'},
    markers=True
)

fig.show()

# For Every Year Show which Country was in the Lead in terms of Total Number of Launches up to and including including 2020)

Do the results change if we only look at the number of successful launches?

In [185]:
# prepare the data
df_lead = df_data.copy()

# Clean date
df_lead['Date'] = pd.to_datetime(df_lead['Date'], errors='coerce')
df_lead = df_lead.dropna(subset=['Date'])

# Extract year
df_lead['Year'] = df_lead['Date'].dt.year

# Extract country
df_lead['Country'] = df_lead['Location'].str.split(',').str[-1].str.strip()

In [186]:
# Limit to ≤ 2020
df_lead = df_lead[df_lead['Year'] <= 2020]

In [187]:
# Leader by Total Launches
total_counts = df_lead.groupby(['Year', 'Country']).size().reset_index(name='Launch_Count')

In [188]:
# Find leader each year
leaders_total = total_counts.loc[
    total_counts.groupby('Year')['Launch_Count'].idxmax()
].sort_values('Year')
# Leader by Successful Launches
df_success = df_lead[df_lead['Mission_Status'] == 'Success']
# Count successful launches
success_counts = df_success.groupby(['Year', 'Country']).size().reset_index(name='Success_Count')
# Find leader each year
leaders_success = success_counts.loc[
    success_counts.groupby('Year')['Success_Count'].idxmax()
].sort_values('Year')

In [189]:
# Compare Results
comparison = leaders_total.merge(
    leaders_success,
    on='Year',
    suffixes=('_Total', '_Success')
)

comparison = comparison[['Year', 'Country_Total', 'Country_Success']]

In [190]:
# Comparison Visualization
fig = px.line(
    total_counts,
    x='Year',
    y='Launch_Count',
    color='Country',
    title='Launch Counts by Country Over Time'
)

fig.show()

# Create a Year-on-Year Chart Showing the Organisation Doing the Most Number of Launches

Which organisation was dominant in the 1970s and 1980s? Which organisation was dominant in 2018, 2019 and 2020?

In [191]:
# Prepare Data
df_org = df_data.copy()

# Clean Date
df_org['Date'] = pd.to_datetime(df_org['Date'], errors='coerce')
df_org = df_org.dropna(subset=['Date'])

# Extract Year
df_org['Year'] = df_org['Date'].dt.year

In [192]:
# Count Launches per Organisation per Year
org_year_counts = df_org.groupby(['Year', 'Organisation']).size().reset_index(name='Launch_Count')

In [193]:
# Find Top Organisation Each Year
leaders = org_year_counts.loc[
    org_year_counts.groupby('Year')['Launch_Count'].idxmax()
].sort_values('Year')

In [194]:
# Plot (Year vs Leading Organisation)
fig = px.scatter(
    leaders,
    x='Year',
    y='Organisation',
    size='Launch_Count',
    color='Organisation',
    title='Top Organisation by Number of Launches Each Year'
)

fig.show()